In [23]:
%pip install requests tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
from pathlib import Path
import pandas as pd
import requests
import time
import json
from tqdm.auto import tqdm

In [9]:
DATASET_ROOT = Path("../dataset")

METADATA_PATH = DATASET_ROOT / "metadata_processed.csv"

RESULTS_DIR = Path("../results/sightengine")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_CSV = RESULTS_DIR / "sightengine_results.csv"

API_USER = "549512133"
API_SECRET = "GXJnpMJ38VzaDwnBJbaozyLPSPeGyc6P"

API_URL = "https://api.sightengine.com/1.0/check.json"

MODEL = "genai"

SLEEP_SECONDS = 0.5
RESULTS_CSV_2 = RESULTS_DIR / "sightengine_results_2.csv"

In [10]:
df = pd.read_csv(METADATA_PATH)

print("Total images:", len(df))
print(df.columns.tolist())

Total images: 2034
['filename', 'filepath', 'label', 'writer_id', 'generator', 'generation_type', 'reference_used', 'reference_based', 'generation_batch', 'split']


In [11]:
test_df = df[df["split"] == "test"].copy()

print("Test images:", len(test_df))
print()
print(test_df["generator"].value_counts())

Test images: 265

generator
Human     149
QN         60
GPT        29
Gemini     27
Name: count, dtype: int64


In [12]:
if RESULTS_CSV.exists():

    results_df = pd.read_csv(RESULTS_CSV)

    completed_paths = set(results_df["path"].astype(str))

    print(f"Existing results: {len(results_df)}")
    print(f"Already completed: {len(completed_paths)}")

else:

    results_df = pd.DataFrame()

    completed_paths = set()

    print("No previous results found. Starting fresh.")

Existing results: 108
Already completed: 83


In [13]:
IMAGE_PATH_COL = "filepath"

print("Using image path column:", IMAGE_PATH_COL)
print("Example:", test_df.iloc[0][IMAGE_PATH_COL])

Using image path column: filepath
Example: ..\processed_dataset\genuine\100.0\100-01.png


In [15]:
# Select images 140–159
# Python index 139 = image 140

selected_df = test_df.iloc[139:].copy()

print("Selected images:", len(selected_df))

print("\nImages 140–159:")
print(selected_df[IMAGE_PATH_COL].tolist())

Selected images: 126

Images 140–159:
['..\\processed_dataset\\genuine\\92.0\\92-01.png', '..\\processed_dataset\\genuine\\92.0\\92-02.png', '..\\processed_dataset\\genuine\\92.0\\92-03.png', '..\\processed_dataset\\genuine\\92.0\\92-04.png', '..\\processed_dataset\\genuine\\92.0\\92-05.png', '..\\processed_dataset\\genuine\\92.0\\92-06.png', '..\\processed_dataset\\genuine\\92.0\\92-07.png', '..\\processed_dataset\\genuine\\92.0\\92-08.png', '..\\processed_dataset\\genuine\\92.0\\92-09.png', '..\\processed_dataset\\genuine\\92.0\\92-10.png', '..\\processed_dataset\\AI\\gemini\\21.0\\image_01.png', '..\\processed_dataset\\AI\\gemini\\21.0\\image_02.png', '..\\processed_dataset\\AI\\gemini\\21.0\\image_03.png', '..\\processed_dataset\\AI\\gemini\\21.0\\image_04.png', '..\\processed_dataset\\AI\\gemini\\21.0\\image_05.png', '..\\processed_dataset\\AI\\gemini\\21.0\\image_06.png', '..\\processed_dataset\\AI\\gemini\\21.0\\image_07.png', '..\\processed_dataset\\AI\\gemini\\21.0\\image_08.p

In [18]:
# Select images 140–159
selected_df = test_df.iloc[139:].copy()

# This is the dataframe used by the API loop
remaining_df = selected_df.copy()

print("Selected images:", len(remaining_df))
print(remaining_df[IMAGE_PATH_COL].tolist())

Selected images: 126
['..\\processed_dataset\\genuine\\92.0\\92-01.png', '..\\processed_dataset\\genuine\\92.0\\92-02.png', '..\\processed_dataset\\genuine\\92.0\\92-03.png', '..\\processed_dataset\\genuine\\92.0\\92-04.png', '..\\processed_dataset\\genuine\\92.0\\92-05.png', '..\\processed_dataset\\genuine\\92.0\\92-06.png', '..\\processed_dataset\\genuine\\92.0\\92-07.png', '..\\processed_dataset\\genuine\\92.0\\92-08.png', '..\\processed_dataset\\genuine\\92.0\\92-09.png', '..\\processed_dataset\\genuine\\92.0\\92-10.png', '..\\processed_dataset\\AI\\gemini\\21.0\\image_01.png', '..\\processed_dataset\\AI\\gemini\\21.0\\image_02.png', '..\\processed_dataset\\AI\\gemini\\21.0\\image_03.png', '..\\processed_dataset\\AI\\gemini\\21.0\\image_04.png', '..\\processed_dataset\\AI\\gemini\\21.0\\image_05.png', '..\\processed_dataset\\AI\\gemini\\21.0\\image_06.png', '..\\processed_dataset\\AI\\gemini\\21.0\\image_07.png', '..\\processed_dataset\\AI\\gemini\\21.0\\image_08.png', '..\\process

In [19]:
for _, row in tqdm(
    remaining_df.iterrows(),
    total=len(remaining_df),
    desc="Sightengine"
):

    image_path = Path(row[IMAGE_PATH_COL])

    if not image_path.exists():
        print(f"\nFile not found: {image_path}")
        continue

    try:

        with open(image_path, "rb") as f:

            files = {
                "media": f
            }

            data = {
                "models": MODEL,
                "api_user": API_USER,
                "api_secret": API_SECRET
            }

            response = requests.post(
                API_URL,
                files=files,
                data=data,
                timeout=60
            )

        result = response.json()

        if result.get("status") == "success":

            ai_score = result.get(
                "type", {}
            ).get(
                "ai_generated",
                None
            )

            operations = result.get(
                "request", {}
            ).get(
                "operations",
                None
            )

            predicted_label = (
                "AI"
                if ai_score is not None and ai_score >= 0.5
                else "Human"
            )

            result_row = {
                "path": str(image_path),
                "true_label": row["label"],
                "generator": row["generator"],
                "writer_id": row.get("writer_id", None),
                "ai_score": ai_score,
                "predicted_label": predicted_label,
                "operations": operations,
                "status": "success"
            }

            one_result_df = pd.DataFrame([result_row])

            # Save to RESULT 2 CSV
            if RESULTS_CSV_2.exists():

                one_result_df.to_csv(
                    RESULTS_CSV_2,
                    mode="a",
                    header=False,
                    index=False
                )

            else:

                one_result_df.to_csv(
                    RESULTS_CSV_2,
                    index=False
                )

            print(
                f"\nSaved: {image_path.name} | "
                f"AI score = {ai_score:.4f}"
            )

        else:

            print(
                f"\nSightengine error for "
                f"{image_path.name}: "
                f"{result.get('error', {})}"
            )

    except Exception as e:

        print(
            f"\nException for "
            f"{image_path.name}: {e}"
        )

    time.sleep(SLEEP_SECONDS)

Sightengine:   0%|          | 0/126 [00:00<?, ?it/s]


Saved: 92-01.png | AI score = 0.0010

Saved: 92-02.png | AI score = 0.0010

Saved: 92-03.png | AI score = 0.0010

Saved: 92-04.png | AI score = 0.0010

Saved: 92-05.png | AI score = 0.0010

Saved: 92-06.png | AI score = 0.0010

Saved: 92-07.png | AI score = 0.0010

Saved: 92-08.png | AI score = 0.0010

Saved: 92-09.png | AI score = 0.0010

Saved: 92-10.png | AI score = 0.0010

Saved: image_01.png | AI score = 0.0010

Saved: image_02.png | AI score = 0.0010

Saved: image_03.png | AI score = 0.0010

Saved: image_04.png | AI score = 0.0010

Saved: image_05.png | AI score = 0.0010

Saved: image_06.png | AI score = 0.0010

Saved: image_07.png | AI score = 0.0010

Saved: image_08.png | AI score = 0.0010

Saved: image_09.png | AI score = 0.0010

Saved: image_01.png | AI score = 0.0010

Saved: image_02.png | AI score = 0.0100

Saved: image_03.png | AI score = 0.0010

Saved: image_04.png | AI score = 0.0010

Saved: image_05.png | AI score = 0.0010

Saved: image_06.png | AI score = 0.0010

Save

In [ ]:
print(test_df.columns.tolist())

['filename', 'filepath', 'label', 'writer_id', 'generator', 'generation_type', 'reference_used', 'reference_based', 'generation_batch', 'split']
